# Source data exploration

A first look at the three raw exports before building anything. This is my attempt at evaluating the data quality of these files.

I decided to use a Jupyter notebook, I find them easier for in-line data analysis!

In [2]:
import pandas as pd

## 1. Stores.csv file 

In [13]:
stores = pd.read_csv('../data/raw/stores.csv')  
print(stores.shape)
stores

(16, 7)


,store_id,store_name,city,state,zip_code,region,opened_date
0,S001,Eastview Mall,Victor,NY,14564,Northeast,2010-03-15
1,S002,Marketplace Mall,Rochester,NY,14623,Northeast,2008-11-01
2,S003,Greece Ridge Center,Greece,NY,938,Northeast,2012-06-20
3,S004,Crossroads Center,St. Cloud,MN,56301,Midwest,2015-09-10
4,S005,Mall of America,Bloomington,MN,55425,Midwest,2011-04-07
5,S006,Lakeside Shopping Ctr,Metairie,LA,70002,South,2009-08-14
6,S007,Downtown Rochester,Rochester,NY,14604,Northeast,2006-01-22
7,S007,Rochester Downtown,Rochester,NY,14604,Northeast,2006-01-22
8,S008,Galleria at Crystal Run,Middletown,NY,10941,Northeast,2014-02-18
9,S009,Tucson Mall,Tucson,AZ,85705,West,2013-07-30


Checking the column types and for missing values.

In [14]:
stores.dtypes

store_id       object
store_name     object
city           object
state          object
zip_code        int64
region         object
opened_date    object
dtype: object

In [15]:
stores.isna().sum()

store_id       0
store_name     0
city           0
state          0
zip_code       0
region         2
opened_date    0
dtype: int64

In [ ]:
# Note: region has null values, looks like they are both portland, OR locations.
stores[stores.region.isna()]

,store_id,store_name,city,state,zip_code,region,opened_date
13,S013,Cascade Station,Portland,OR,97220,NaN,2019-05-14
14,S014,Lloyd Center,Portland,OR,97232,NaN,2007-09-09


In [ ]:
# This doesn't seem to be a huge issue because region looks like it can be easily inferred based on the state. 
# This would show up in a simple null count check in our profiler, but for cleaning, i think we can infer that OR would be region = 'West'
stores.groupby('state').region.unique()

state
AZ         [West]
LA        [South]
MN      [Midwest]
NY    [Northeast]
OR          [nan]
TX        [South]
WA         [West]
Name: region, dtype: object

`store_id` should be unique. Checking duplicate rows and duplicate IDs separately.

In [18]:
print('duplicate rows:', stores.duplicated().sum())
print('duplicate store_ids:', stores.store_id.duplicated().sum())

duplicate rows: 0
duplicate store_ids: 1


In [ ]:
# Note: this would be missed by a simple duplicate count check, but if we check the primary key we find it. Our profiler needs to check each primary key as well for duplicates.
stores[stores.store_id.duplicated(keep=False)]

,store_id,store_name,city,state,zip_code,region,opened_date
6,S007,Downtown Rochester,Rochester,NY,14604,Northeast,2006-01-22
7,S007,Rochester Downtown,Rochester,NY,14604,Northeast,2006-01-22


`zip_code` was read as a number. i noticed above that one of the zipcodes was not 5 digits. This should be flagged.

In [20]:
stores.zip_code.astype(str).str.len().value_counts()

5    15
3     1
Name: zip_code, dtype: int64

When we read `zip_code` as text to see the value as it appears in the file, we have a leading 0 that was dropped. Zip codes should be 5 digits, still. I'll have to decide what to do with this later.

In [22]:
raw_zip = pd.read_csv('../data/raw/stores.csv', dtype={'zip_code': str})
raw_zip[raw_zip.zip_code.str.len() != 5]

,store_id,store_name,city,state,zip_code,region,opened_date
2,S003,Greece Ridge Center,Greece,NY,0938,Northeast,2012-06-20


In [ ]:
#no issues with the date. Seems to be all in the same format and type.
pd.to_datetime(stores.opened_date, format='%Y-%m-%d').agg(['min', 'max'])

min   2006-01-22
max   2020-01-15
Name: opened_date, dtype: datetime64[ns]

**Stores findings**
- `region` is blank for both Portland, OR stores (S013, S014). No other OR store exists to copy from. (but I think we can infer the region based on the state)
- `S007` appears twice with different names ("Downtown Rochester" / "Rochester Downtown"). Every other field matches. (I think we can drop one of these, they are identical besides the name ordering)
- S003 has zip `0938`: only 4 digits, and it would load as `938` if read as a number. Greece, NY zips start with 146. (these are not valid zips, need to consider about how to handle these in the cleaning step)
- `opened_date` parses cleanly.

## 2. Products

In [25]:
products = pd.read_csv('../data/raw/products.csv')
print(products.shape)
products

(32, 5)


,product_id,product_name,category,unit_price,supplier_id
0,P030,Product P030,Office Supplies,76.48,SUP005
1,P016,Product P016,NaN,24.52,SUP001
2,P025,Product P025,Office Supplies,140.26,SUP005
3,P018,Product P018,Home & Garden,107.56,SUP003
4,P009,Product P009,NaN,67.73,SUP004
5,P010,Product P010,Office Supplies,237.08,SUP005
6,P005,Product P005,Office Supplies,150.11,SUP005
7,P026,Product P026,Electronics,54.13,SUP001
8,P013,Product P013,Home & Garden,37.22,SUP003
9,P001,Product P001,Electronics,157.39,SUP001


In [26]:
products.dtypes

product_id       object
product_name     object
category         object
unit_price      float64
supplier_id      object
dtype: object

In [27]:
products.isna().sum()

product_id      0
product_name    0
category        5
unit_price      0
supplier_id     0
dtype: int64

In [ ]:

products[products.category.isna()]

,product_id,product_name,category,unit_price,supplier_id
1,P016,Product P016,NaN,24.52,SUP001
4,P009,Product P009,NaN,67.73,SUP004
17,P003,Product P003,NaN,134.20,SUP003
22,P023,Product P023,NaN,108.28,SUP003
29,P029,Product P029,NaN,155.01,SUP004


Missign categories - but it seems like supplier may link up 1 to 1 with category. Looks like we can replace missing Nulls in the cleaning step using it.

In [29]:
pd.crosstab(products.supplier_id, products.category)

category,Apparel,Electronics,Food & Beverage,Home & Garden,Office Supplies
supplier_id,,,,,
SUP001,0,5,0,0,0
SUP002,7,0,0,0,0
SUP003,0,0,0,4,0
SUP004,0,0,4,0,0
SUP005,0,0,0,0,7


Checking duplicate rows and duplicate `product_id`s.

In [30]:
print('duplicate rows:', products.duplicated().sum())
print('duplicate product_ids:', products.product_id.duplicated().sum())

duplicate rows: 1
duplicate product_ids: 2


In [ ]:
#again, like stores, we can see that there are some duplicates primary keys
products[products.product_id.duplicated(keep=False)].sort_values('product_id')

,product_id,product_name,category,unit_price,supplier_id
6,P005,Product P005,Office Supplies,150.11,SUP005
10,P005,Product P005,Office Supplies,141.61,SUP005
14,P012,Product P012,Apparel,123.17,SUP002
24,P012,Product P012,Apparel,123.17,SUP002


In [32]:
products.describe()

,unit_price
count,32.000000
mean,119.165625
std,72.883373
min,0.000000
25%,64.330000
50%,125.065000
75%,155.605000
max,245.840000


In [34]:
products[products.unit_price == 0]

,product_id,product_name,category,unit_price,supplier_id
18,P027,Product P027,Apparel,0.0,SUP002


**Products findings**
- 5 products have no `category`. Each supplier maps to exactly one category, so the blanks can be filled in from `supplier_id`.
- `P012` is an exact duplicate row. `P005` appears twice with **different prices** (150.11 vs 141.61).
- `P027` has a catalog price of 0.00. (i doubt that this product is free! This is something I would bring up with the client, so I'll have to figure out how to handle it in the meantime.)
- `product_name` is just "Product " + ID, so it carries no extra information.
- The catalog covers P001–P030. (maybe we can set this as a range during cleaning, but it would throw an error if new catalog items are added)

## 3. Transactions

In [ ]:
tx = pd.read_csv('../data/raw/transactions.csv')
print(tx.shape)
tx.head(100)

#This data is a big longer at 505 records, harder to read manually by just querying the table.

(505, 8)


,transaction_id,transaction_date,store_id,product_id,customer_id,quantity,unit_price,total_amount
0,TXN10179,2026-05-30,S011,P024,CUST0247,2,196.20,392.4
1,TXN10209,2026-04-23,S015,P027,NaN,4,195.34,781.36
2,TXN10385,2026-03-11,S014,P030,CUST0059,4,76.48,305.92
3,TXN10130,2026-04-17,S011,P010,CUST0060,5,237.08,1185.4
4,TXN20016,2026-05-25,S007,P030,CUST0236,-2,76.48,-152.96
...,...,...,...,...,...,...,...,...
95,TXN10431,2026-05-12,S001,P019,CUST0188,4,193.38,773.52
96,TXN10432,2026-04-30,S014,P011,CUST0186,1,224.93,224.93
97,TXN10171,2026-05-13,S011,P002,CUST0207,2,147.38,294.76
98,TXN10265,2026-04-14,S002,P020,CUST0012,1,26.47,26.47


In [39]:
tx.dtypes

transaction_id       object
transaction_date     object
store_id             object
product_id           object
customer_id          object
quantity              int64
unit_price          float64
total_amount         object
dtype: object

Looks like`total_amount` didn't load as a number, and `transaction_date` is text. Need to investigate these *

### 3a. Missing values

In [40]:
tx.isna().sum()

transaction_id       0
transaction_date     0
store_id             0
product_id           0
customer_id         40
quantity             0
unit_price           0
total_amount         0
dtype: int64

In [41]:
tx[tx.customer_id.isna()].head()

,transaction_id,transaction_date,store_id,product_id,customer_id,quantity,unit_price,total_amount
1,TXN10209,2026-04-23,S015,P027,NaN,4,195.34,781.36
13,TXN10235,2026-05-18,S001,P011,NaN,5,224.93,1124.65
14,TXN10239,2026-03-27,S003,P007,NaN,3,104.92,314.76
22,TXN10211,2026-05-31,S011,P018,NaN,2,107.56,215.12
35,TXN10221,2026-03-19,S004,P011,NaN,1,224.93,224.93


### 3b. Duplicates

In [43]:
print('duplicate rows:', tx.duplicated().sum())
print('duplicate transaction_ids:', tx.transaction_id.duplicated().sum())

duplicate rows: 15
duplicate transaction_ids: 15


In [ ]:
# the duplicate rows are simple enough, we can dedup pretty easily here.
tx[tx.duplicated(keep=False)].sort_values('transaction_id')

,transaction_id,transaction_date,store_id,product_id,customer_id,quantity,unit_price,total_amount
347,TXN10051,2026-04-14,S003,P024,CUST0224,2,196.20,392.4
335,TXN10051,2026-04-14,S003,P024,CUST0224,2,196.20,392.4
331,TXN10052,2026-03-31,S005,P005,CUST0069,2,141.61,283.22
299,TXN10052,2026-03-31,S005,P005,CUST0069,2,141.61,283.22
173,TXN10053,2026-05-09,S011,P004,CUST0044,5,10.53,52.65
201,TXN10053,2026-05-09,S011,P004,CUST0044,5,10.53,52.65
290,TXN10054,2026-04-15,S009,P027,CUST0093,4,195.34,781.36
174,TXN10054,2026-04-15,S009,P027,CUST0093,4,195.34,781.36
80,TXN10055,2026-03-21,S013,P027,CUST0093,3,195.34,586.02
329,TXN10055,2026-03-21,S013,P027,CUST0093,3,195.34,586.02


### 3c. Quantities

In [ ]:
#we do have negative vaues here - I know that the instructions read that we should consider negative value are returns *
tx.describe()

,quantity,unit_price
count,505.000000,505.000000
mean,2.623762,123.835069
std,2.040520,70.675164
min,-5.000000,10.530000
25%,1.000000,69.620000
50%,3.000000,126.960000
75%,4.000000,193.380000
max,5.000000,245.840000


`total_amount` is missing from `describe()` because it isn't numeric. Quantity runs from -5 to 5. Looking at zeros and negatives.

In [48]:
tx[tx.quantity == 0]

,transaction_id,transaction_date,store_id,product_id,customer_id,quantity,unit_price,total_amount
43,TXN10251,2026-04-07,S001,P025,CUST0037,0,140.26,0
282,TXN10253,2026-05-19,S007,P001,CUST0236,0,157.39,0
327,TXN10252,2026-04-23,S002,P022,CUST0249,0,216.88,0
386,TXN10254,2026-03-19,S004,P022,CUST0272,0,216.88,0
396,TXN10255,2026-05-14,S007,P004,CUST0181,0,10.53,0


In [73]:
tx[tx.quantity < 0].head()

,transaction_id,transaction_date,store_id,product_id,customer_id,quantity,unit_price,total_amount
4,TXN20016,2026-05-25,S007,P030,CUST0236,-2,76.48,-152.96
5,TXN20019,2026-04-01,S006,P002,CUST0168,-1,147.38,-147.38
16,TXN20004,2026-05-16,S013,P028,CUST0133,-5,69.62,-348.1
40,TXN20009,2026-04-18,S011,P023,CUST0136,-1,108.28,-108.28
45,TXN20030,2026-04-19,S015,P028,CUST0141,-3,69.62,-208.86


The negative rows all seem to have IDs starting `TXN2`. Checking quantity range by ID prefix:

In [50]:
tx.groupby(tx.transaction_id.str[:4]).quantity.agg(['count', 'min', 'max'])

,count,min,max
transaction_id,,,
TXN1,475,0,5
TXN2,30,-5,-1


If these are returns, it would follow that each should match an earlier sale with the same store, product, customer and date.

Note: The code from this section is from LLM log #1 in the transcript file.

In [52]:
# 1. Does every return have a matching sale?
m = returns.merge(sales[keys].drop_duplicates(), on=keys, how='left', indicator=True)
print(m['_merge'].value_counts())          # 'left_only' = return with no matching sale
unmatched = m[m['_merge'] == 'left_only']

both          30
left_only      0
right_only     0
Name: _merge, dtype: int64


In [53]:
# 2. Do returns ever exceed what was sold for that store/product/customer/date?
agg = (tx.assign(sold=tx['quantity'].clip(lower=0),
                 returned=-tx['quantity'].clip(upper=0))
         .groupby(keys, dropna=False)[['sold', 'returned']].sum())
over_returned = agg[agg['returned'] > agg['sold']]
print(len(over_returned), "groups where returns exceed sales")

0 groups where returns exceed sales


In [54]:
# 3. Does the return come after the sale? (assumes transaction_id increases over time)
first_sale = sales.groupby(keys, dropna=False)['transaction_id'].min().rename('first_sale_id')
chk = returns.join(first_sale, on=keys)
print((chk['transaction_id'] < chk['first_sale_id']).sum(), "returns logged before their sale")

0 returns logged before their sale


### 3d. Total amount

In [58]:
amount_try = pd.to_numeric(tx.total_amount, errors='coerce')
print('values that are not numbers:', amount_try.isna().sum())
tx.loc[amount_try.isna(), ['transaction_id', 'total_amount']]

values that are not numbers: 25


,transaction_id,total_amount
12,TXN10041,$615.85
19,TXN10033,$134.20
21,TXN10038,$1185.40
33,TXN10031,$380.88
39,TXN10032,$507.84
68,TXN10024,$348.10
100,TXN10028,$67.73
112,TXN10025,$536.80
123,TXN10035,$899.72
134,TXN10034,$12.81


### 3e. Transaction dates

Parsing strictly as YYYY-MM-DD to see what doesn't fit.

In [61]:
iso = pd.to_datetime(tx.transaction_date, format='%Y-%m-%d', errors='coerce')
print("dates that aren't YYYY-MM-DD:", iso.isna().sum())
tx.loc[iso.isna(), ['transaction_id', 'transaction_date']]

dates that aren't YYYY-MM-DD: 20


,transaction_id,transaction_date
28,TXN10008,03/19/2026
50,TXN10011,01-04-2026
90,TXN10013,24-03-2026
133,TXN10001,04/12/2026
164,TXN10009,04/01/2026
194,TXN10002,05/07/2026
199,TXN10005,04/16/2026
206,TXN10020,12-03-2026
234,TXN10018,15-04-2026
240,TXN10007,04/30/2026


Two other formats appear: `NN/NN/YYYY` and `NN-NN-YYYY`. Which part is the day and which is the month? A value above 12 can only be a day.

In [62]:
odd = tx.loc[iso.isna(), ['transaction_id', 'transaction_date']].copy()
odd['sep'] = odd.transaction_date.str[2]
odd['first'] = odd.transaction_date.str[:2].astype(int)
odd['second'] = odd.transaction_date.str[3:5].astype(int)
odd.groupby('sep')[['first', 'second']].max()

,first,second
sep,,
-,26,5
/,5,30


With `/` the second part goes above 12, so it's month first. With `-` the first part goes above 12, so it's day first. Some values could still be read either way:

In [63]:
odd[(odd['first'] <= 12) & (odd['second'] <= 12)]

,transaction_id,transaction_date,sep,first,second
50,TXN10011,01-04-2026,-,1,4
133,TXN10001,04/12/2026,/,4,12
164,TXN10009,04/01/2026,/,4,1
194,TXN10002,05/07/2026,/,5,7
206,TXN10020,12-03-2026,-,12,3
294,TXN10006,05/05/2026,/,5,5
447,TXN10017,06-04-2026,-,6,4


Now the range of the clean YYYY-MM-DD dates, by month:

In [65]:
print(iso.min(), '->', iso.max())
iso.dt.to_period('M').value_counts().sort_index()

2026-03-05 00:00:00 -> 2026-06-27 00:00:00


2026-03    124
2026-04    156
2026-05    198
2026-06      7
Freq: M, Name: transaction_date, dtype: int64

**Transactions findings**
- 40 transactions have no `customer_id`, all in one ID block (TXN10201–TXN10240). This looks like guest checkout, not lost data.
- 15 rows are exact duplicates. Every duplicate `transaction_id` is an exact copy, so no two versions conflict.
- 5 transactions have quantity 0 and total 0.
- The 30 negative-quantity rows are returns: all `TXN2…` IDs, each matching a same-day sale for the full quantity.
- 25 `total_amount` values have a `$` in front, which stops the column loading as a number. We'll have to strip these during cleaning
- 20 dates aren't YYYY-MM-DD: 10 are MM/DD/YYYY (TXN10001–TXN10010) and 10 are DD-MM-YYYY (TXN10011–TXN10020). 7 of them could be read either way.

## 4. How the files fit together

Does every transaction point to a store and product that exist?

In [68]:
tx[~tx.store_id.isin(stores.store_id)]

,transaction_id,transaction_date,store_id,product_id,customer_id,quantity,unit_price,total_amount
149,TXN10151,2026-05-04,S016,P017,CUST0054,4,126.96,507.84
166,TXN10153,2026-05-06,S018,P020,CUST0020,4,26.47,105.88
212,TXN10152,2026-04-17,S017,P021,CUST0047,3,129.12,387.36
338,TXN10154,2026-05-02,S016,P030,CUST0141,2,76.48,152.96
423,TXN10155,2026-04-29,S019,P013,CUST0245,3,37.22,111.66


In [69]:
tx[~tx.product_id.isin(products.product_id)]

,transaction_id,transaction_date,store_id,product_id,customer_id,quantity,unit_price,total_amount
94,TXN10157,2026-05-02,S008,P032,CUST0229,1,242.36,242.36
228,TXN10158,2026-05-19,S006,P031,CUST0254,2,123.17,246.34
487,TXN10156,2026-05-14,S001,P031,CUST0244,2,157.39,314.78


Does the price charged match the catalog price?

In [70]:
sold = tx[['product_id', 'unit_price']].drop_duplicates()
catalog = products[['product_id', 'unit_price']].drop_duplicates()

prices = sold.merge(catalog, on='product_id', how='left', suffixes=('_sold', '_catalog'))
prices[prices.unit_price_sold != prices.unit_price_catalog]

,product_id,unit_price_sold,unit_price_catalog
1,P027,195.34,0.00
27,P005,141.61,150.11
31,P032,242.36,NaN
32,P031,123.17,NaN
33,P031,157.39,NaN


**Cross-file findings**
- 5 transactions use stores S016–S019, which aren't in the store list.
- 3 transactions use products P031 and P032, which aren't in the catalog. P031 sold at two different prices.
- P027 sells at 195.34 against a catalog price of 0.00. (I think we can consider updating the catalog price based on this price)
- P005 always sells at 141.61, which suggests that is the correct one of its two catalog rows. (we can update this as well)